In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from faker import Faker
import random
import os

# ── INITIAL SETUP ───────────────────────────────────────
fake = Faker()

np.random.seed(42)
random.seed(42)

# ── CONFIG ──────────────────────────────────────────────
START_DATE = "2021-01-01"
END_DATE = "2023-12-31"
N_TRANSACTIONS = 15000

REGIONS = ["North", "South", "East", "West", "Central"]

PRODUCTS = {
    "Electronics": (15000, 150000),
    "Clothing":    (500, 5000),
    "Food":        (100, 1500),
    "Furniture":   (5000, 200000),
    "Software":    (2000, 25000),
}

CHANNELS = ["Online", "In-Store", "Partner", "Direct"]

# ── FUNCTION TO GENERATE SALES DATA ────────────────────
def generate_sales_data(n):
    """Generate realistic sales transactions."""

    dates = pd.date_range(
        start=START_DATE,
        end=END_DATE,
        freq="D"
    )

    records = []

    for i in range(n):

        # Convert to pandas Timestamp
        date = pd.to_datetime(np.random.choice(dates))

        # Random product category
        category = np.random.choice(
            list(PRODUCTS.keys()),
            p=[0.25, 0.20, 0.20, 0.15, 0.20]
        )

        lo, hi = PRODUCTS[category]

        # Q4 seasonal sales boost
        seasonal = 1.3 if date.month in [10, 11, 12] else 1.0

        # Year-over-year growth
        yoy = 1.0 + (0.08 * (date.year - 2021))

        # Pricing and metrics
        unit_price = round(
            np.random.uniform(lo, hi) * yoy,
            2
        )

        quantity = np.random.randint(1, 20)

        revenue = round(
            unit_price * quantity * seasonal,
            2
        )

        cost = round(
            revenue * np.random.uniform(0.45, 0.70),
            2
        )

        profit = round(revenue - cost, 2)

        # Create transaction record
        records.append({
            "transaction_id": f"TXN-{i+1:05d}",
            "date": date,
            "year": date.year,
            "quarter": f"Q{date.quarter}",
            "month": date.month,
            "month_name": date.strftime("%b"),
            "region": np.random.choice(REGIONS),
            "category": category,
            "channel": np.random.choice(CHANNELS),
            "customer_id": f"CUST-{np.random.randint(1000, 5000):04d}",
            "unit_price": unit_price,
            "quantity": quantity,
            "revenue": revenue,
            "cost": cost,
            "profit": profit,
        })

    return pd.DataFrame(records)

# ── GENERATE DATA ──────────────────────────────────────
df = generate_sales_data(N_TRANSACTIONS)

# Sort by date
df = df.sort_values("date").reset_index(drop=True)

# ── CREATE FOLDER ──────────────────────────────────────
os.makedirs("../data/raw", exist_ok=True)

# ── SAVE CSV ───────────────────────────────────────────
file_path = "../data/raw/sales_data.csv"

df.to_csv(file_path, index=False)

# ── OUTPUT ─────────────────────────────────────────────
print("=" * 50)
print(f"✅ Generated {len(df):,} transactions")
print("=" * 50)

print("\n📌 First 5 Rows:")
print(df.head())

print("\n📌 Data Types:")
print(df.dtypes)

print("\n📌 CSV Saved Successfully:")
print(file_path)

✅ Generated 15,000 transactions

📌 First 5 Rows:
  transaction_id       date  year quarter  month month_name region  \
0      TXN-09965 2021-01-01  2021      Q1      1        Jan  North   
1      TXN-02163 2021-01-01  2021      Q1      1        Jan  North   
2      TXN-01225 2021-01-01  2021      Q1      1        Jan   West   
3      TXN-04023 2021-01-01  2021      Q1      1        Jan   West   
4      TXN-05533 2021-01-01  2021      Q1      1        Jan   West   

      category   channel customer_id  unit_price  quantity    revenue  \
0     Clothing   Partner   CUST-2609     2365.22         9   21286.98   
1  Electronics    Online   CUST-1582    91994.88         2  183989.76   
2     Clothing   Partner   CUST-4862     1324.39         5    6621.95   
3     Clothing  In-Store   CUST-3955     1760.93         8   14087.44   
4     Software  In-Store   CUST-3286    14746.12         2   29492.24   

        cost    profit  
0   13545.88   7741.10  
1  114503.21  69486.55  
2    3775.58   2